In [1]:
%pip install pulp

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.1/16.4 MB 787.7 kB/s eta 0:00:21
   ---------------------------------------- 0.2/16.4 MB 1.3 MB/s eta 0:00:13
   ---------------------------------------- 0.2/16.4 MB 1.3 MB/s eta 0:00:13
    --------------------------------------- 0.3/16.4 MB 1.4 MB/s eta 0:00:12
    --------------------------------------- 0.3/16.4 MB 1.4 MB/s eta 0:00:12
    --------------------------------------- 0.3/16.4 MB 999.9 kB/s eta 0:00:17
   - -------------------------------------- 0.4/16.4 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.5/16.4 MB 1.2 MB/s eta 0:00:13
   - -------------------------------------- 0.6/16.4 MB 1.3 MB/s eta 0:00:12
   - -------------------------------------- 0.8/16.4 MB 1.6 MB/s eta 0:00:10
   -- ----------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pulp

print("PuLP version:", pulp.__version__)

PuLP version: 3.3.2


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_supply_chain.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (180519, 51)


In [4]:
scenario = {
    "shipment_id": "SC-1001",
    "required_units": 100,
    "available_budget": 20000,
    "maximum_acceptable_delay": 7
}

print("Supply Chain Scenario")
print("=" * 30)

for key, value in scenario.items():
    print(f"{key}: {value}")

Supply Chain Scenario
shipment_id: SC-1001
required_units: 100
available_budget: 20000
maximum_acceptable_delay: 7


In [5]:
actions = {
    "Air Freight": {
        "cost": 15000,
        "delay_days": 2,
        "capacity": 150
    },
    "Secondary Supplier": {
        "cost": 12500,
        "delay_days": 5,
        "capacity": 120
    },
    "Delay Product Launch": {
        "cost": 5000,
        "delay_days": 14,
        "capacity": 100
    }
}

for action, details in actions.items():
    print(f"\n{action}")
    print(f"  Cost: £{details['cost']:,}")
    print(f"  Delay: {details['delay_days']} days")
    print(f"  Capacity: {details['capacity']} units")


Air Freight
  Cost: £15,000
  Delay: 2 days
  Capacity: 150 units

Secondary Supplier
  Cost: £12,500
  Delay: 5 days
  Capacity: 120 units

Delay Product Launch
  Cost: £5,000
  Delay: 14 days
  Capacity: 100 units


In [6]:
feasible_actions = {}

for action, details in actions.items():

    within_budget = details["cost"] <= scenario["available_budget"]
    within_delay = details["delay_days"] <= scenario["maximum_acceptable_delay"]
    enough_capacity = details["capacity"] >= scenario["required_units"]

    if within_budget and within_delay and enough_capacity:
        feasible_actions[action] = details

print("Feasible actions:")
print("=" * 30)

for action, details in feasible_actions.items():
    print(
        f"{action}: "
        f"£{details['cost']:,}, "
        f"{details['delay_days']} days, "
        f"{details['capacity']} units"
    )

Feasible actions:
Air Freight: £15,000, 2 days, 150 units
Secondary Supplier: £12,500, 5 days, 120 units


In [7]:
# Create a binary decision variable for each action
decision_vars = {
    action: pulp.LpVariable(
        action.replace(" ", "_"),
        cat="Binary"
    )
    for action in actions
}

# Create the optimization problem
problem = pulp.LpProblem(
    "Supply_Prescript_Optimization",
    pulp.LpMinimize
)

# Objective: minimize total decision cost
problem += pulp.lpSum(
    actions[action]["cost"] * decision_vars[action]
    for action in actions
)

# Constraint 1: choose exactly ONE action
problem += pulp.lpSum(
    decision_vars[action]
    for action in actions
) == 1

# Constraint 2: total cost must stay within budget
problem += pulp.lpSum(
    actions[action]["cost"] * decision_vars[action]
    for action in actions
) <= scenario["available_budget"]

# Constraint 3: selected action must meet acceptable delay
problem += pulp.lpSum(
    actions[action]["delay_days"] * decision_vars[action]
    for action in actions
) <= scenario["maximum_acceptable_delay"]

# Constraint 4: selected action must have enough capacity
problem += pulp.lpSum(
    actions[action]["capacity"] * decision_vars[action]
    for action in actions
) >= scenario["required_units"]

print("Optimization model created successfully!")

Optimization model created successfully!


In [8]:
# Solve the optimization problem
problem.solve(pulp.PULP_CBC_CMD(msg=False))

print("Solver status:", pulp.LpStatus[problem.status])

Solver status: Optimal


In [9]:
# Identify the selected action

selected_action = None

for action, variable in decision_vars.items():
    if variable.value() == 1:
        selected_action = action
        break

print("Recommended Action:", selected_action)
print("Recommended Cost: £{:,.0f}".format(
    actions[selected_action]["cost"]
))
print("Expected Delay:", actions[selected_action]["delay_days"], "days")
print("Available Capacity:", actions[selected_action]["capacity"], "units")

Recommended Action: Secondary Supplier
Recommended Cost: £12,500
Expected Delay: 5 days
Available Capacity: 120 units
